<a href="https://colab.research.google.com/github/seihoon/workeraction/blob/main/%EB%B0%98%EB%8F%84%EC%B2%B4_%EA%B3%B5%EC%A0%95_%EB%8D%B0%EC%9D%B4%ED%84%B0%EB%B6%84%EC%84%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 반도체 공정 데이터 분석 실습
### Pandas를 활용한 데이터 분석 A to Z

> 이 노트북은 **반도체 공정 센서 데이터**를 분석하여  
> 제품의 합격/불합격 여부를 예측하는 과정을 단계별로 배웁니다.

---

## 📚 학습 목차

| 단계 | 내용 | 핵심 함수 |
|:----:|------|-----------|
| 1️⃣ | 환경 설정 & 라이브러리 | `import` |
| 2️⃣ | 데이터 불러오기 & 기본 탐색 | `read_csv()`, `head()`, `info()`, `describe()` |
| 3️⃣ | 결측값 처리 | `isnull()`, `fillna()` |
| 4️⃣ | 탐색적 데이터 분석 (EDA) | `groupby()`, `corr()` |
| 5️⃣ | 데이터 시각화 | `matplotlib`, `seaborn` |
| 6️⃣ | 머신러닝 모델 만들기 | `scikit-learn` |
| 7️⃣ | 모델 평가 & 해석 | `confusion_matrix`, `feature_importance` |

---


## 1️⃣ 환경 설정 & 라이브러리 불러오기

> **라이브러리(Library)** 란?  
> 다른 사람들이 미리 만들어둔 유용한 코드 묶음입니다.  
> 우리는 이걸 `import` 해서 바로 사용할 수 있어요!

| 라이브러리 | 역할 |
|-----------|------|
| `pandas` | 표(DataFrame) 형태로 데이터를 다루는 도구 |
| `numpy` | 수학/통계 계산 도구 |
| `matplotlib` | 그래프를 그리는 기본 도구 |
| `seaborn` | 더 예쁜 통계 그래프 도구 |
| `scikit-learn` | 머신러닝 모델 도구 |


In [ ]:
# ── 코랩 한글 폰트 설치 (최초 1회만 실행) ──────────────────
# 주석(#)을 지우고 실행하세요. 완료 후 런타임 재시작 필요!
# !apt-get install -y fonts-nanum > /dev/null 2>&1
# !fc-cache -fv > /dev/null 2>&1

# ── 라이브러리 불러오기 ──────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, auc,
                             accuracy_score)
import warnings
warnings.filterwarnings('ignore')

# ── 그래프 스타일 설정 ───────────────────────────────────────
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'NanumGothic'   # 한글 폰트
plt.rcParams['axes.unicode_minus'] = False     # 마이너스 기호 깨짐 방지
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette("muted")

print("✅ 모든 라이브러리 로딩 완료!")


---
## 2️⃣ 데이터 불러오기 & 기본 탐색

> ### 📌 데이터 컬럼 설명
> | 컬럼명 | 의미 | 단위 |
> |--------|------|------|
> | 측정시간 | 센서 측정 시각 | - |
> | 온도_섭씨 | 공정 챔버 내부 온도 | °C |
> | 압력_Pa | 챔버 내부 압력 | Pa |
> | 가스유량_slm | 공정 가스 유량 | slm |
> | 전력_W | 인가 전력 | W |
> | 진공도_mTorr | 진공도 | mTorr |
> | 두께_nm | 증착된 박막 두께 | nm |
> | 습도_pct | 공정실 습도 | % |
> | 진동_mm_s | 장비 진동 강도 | mm/s |
> | 처리시간_sec | 공정 처리 시간 | 초 |
> | 냉각수온도_섭씨 | 냉각수 온도 | °C |
> | **합격여부** | **1=합격, -1=불합격** | - |


In [ ]:
# ── CSV 파일 불러오기 ────────────────────────────────────────
# pd.read_csv() : CSV 파일을 읽어 DataFrame으로 만들어 줍니다
df = pd.read_csv('반도체_공정_샘플.csv')

# ── 기본 정보 출력 ───────────────────────────────────────────
print(f"📊 데이터 크기  : {df.shape[0]}행(row) × {df.shape[1]}열(column)")
print(f"📋 컬럼 목록   : {df.columns.tolist()}")
print(f"\n🔍 합격여부 분포:")
vc = df['합격여부'].value_counts()
for k, v in vc.items():
    label = '✅ 합격' if k == 1 else '❌ 불합격'
    print(f"   {label}({k}): {v}개 ({v/len(df)*100:.1f}%)")


In [ ]:
# ── 처음 5행 미리보기 ───────────────────────────────────────
# head(n) : 위에서 n행 보여주기 (기본값=5)
print("📋 데이터 상위 5행:")
df.head()


In [ ]:
# ── 마지막 5행 보기 ─────────────────────────────────────────
# tail(n) : 아래에서 n행 보여주기
print("📋 데이터 하위 5행:")
df.tail()


In [ ]:
# ── 데이터 타입 & 메모리 정보 ───────────────────────────────
# info() : 각 컬럼의 데이터 타입과 비결측 개수를 보여줍니다
df.info()


In [ ]:
# ── 기술 통계 요약 ──────────────────────────────────────────
# describe() : 평균·표준편차·사분위수 등을 자동 계산해 줍니다
# 💡 count가 200보다 작으면 결측값이 있다는 신호!
df.describe().round(2)


In [ ]:
# ── 합격여부 분포 시각화 ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 왼쪽: 막대그래프
colors = ['#2ecc71', '#e74c3c']
counts = df['합격여부'].value_counts().sort_index()
labels = ['불합격(-1)', '합격(1)']
bars = axes[0].bar(labels, [counts[-1], counts[1]],
                   color=['#e74c3c', '#2ecc71'], width=0.5,
                   edgecolor='white', linewidth=2)
axes[0].set_title('합격 / 불합격 건수', fontsize=13, fontweight='bold')
axes[0].set_ylabel('건수')
for bar, val in zip(bars, [counts[-1], counts[1]]):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 1,
                 f'{val}건', ha='center', fontweight='bold', fontsize=12)

# 오른쪽: 파이차트
pie_vals = [counts[-1], counts[1]]
pie_labels = [f'불합격\n{counts[-1]}건 ({counts[-1]/200*100:.0f}%)',
              f'합격\n{counts[1]}건 ({counts[1]/200*100:.0f}%)']
axes[1].pie(pie_vals, labels=pie_labels,
            colors=['#e74c3c', '#2ecc71'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('합격률 비율', fontsize=13, fontweight='bold')

plt.suptitle('📊 합격여부 분포', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print(f"\n💡 합격률: {counts[1]/200*100:.1f}%  |  불합격률: {counts[-1]/200*100:.1f}%")


---
## 3️⃣ 결측값(Missing Value) 처리

> **결측값**이란 측정 오류, 기록 누락 등으로 **데이터가 비어있는 칸**입니다.  
> 머신러닝 모델은 결측값을 그대로 처리할 수 없으므로 반드시 처리해야 합니다!

### 결측값 처리 전략
| 방법 | 설명 | 언제 사용? |
|------|------|-----------|
| **평균(mean)으로 채우기** | 해당 컬럼의 평균값 | 정규분포에 가까울 때 |
| **중앙값(median)으로 채우기** | 이상치에 강인함 | 이상치가 많을 때 |
| **최빈값(mode)으로 채우기** | 가장 자주 나온 값 | 범주형 데이터 |
| **행 삭제** | 결측값 있는 행 제거 | 결측 비율이 매우 낮을 때 |


In [ ]:
# ── 결측값 개수 & 비율 확인 ─────────────────────────────────
missing_count = df.isnull().sum()
missing_pct   = (missing_count / len(df) * 100).round(1)

missing_df = pd.DataFrame({
    '결측값 개수': missing_count,
    '결측률(%)' : missing_pct
}).query('`결측값 개수` > 0').sort_values('결측률(%)', ascending=False)

print("📋 결측값 현황:")
print(missing_df.to_string())
print(f"\n⚠️  전체 결측값 합계: {df.isnull().sum().sum()}개")


In [ ]:
# ── 결측값 히트맵 시각화 ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 결측률 막대그래프
cols_with_na = missing_df.index.tolist()
pcts = missing_df['결측률(%)'].values
colors_bar = ['#e74c3c' if p > 7 else '#f39c12' if p > 4 else '#3498db'
              for p in pcts]
bars = axes[0].barh(cols_with_na, pcts, color=colors_bar,
                    edgecolor='white', height=0.6)
axes[0].set_xlabel('결측률 (%)')
axes[0].set_title('컬럼별 결측률', fontsize=13, fontweight='bold')
for bar, val in zip(bars, pcts):
    axes[0].text(val + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{val}%', va='center', fontsize=10)
axes[0].set_xlim(0, 12)

# 범례
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', label='> 7%'),
                   Patch(facecolor='#f39c12', label='4~7%'),
                   Patch(facecolor='#3498db', label='< 4%')]
axes[0].legend(handles=legend_elements, title='결측률 수준', loc='lower right')

# 오른쪽: 결측값 위치 히트맵
numeric_cols_na = df.select_dtypes(include='number').columns.tolist()
missing_matrix = df[numeric_cols_na].isnull().astype(int)
sns.heatmap(missing_matrix.T, cmap=['#ecf0f1', '#e74c3c'],
            cbar=False, ax=axes[1], linewidths=0.3,
            xticklabels=False)
axes[1].set_title('결측값 위치 지도\n(빨강=결측)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('데이터 행 번호')

plt.suptitle('🔍 결측값 분석', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── 결측값 처리: 평균값으로 채우기 ─────────────────────────
# 💡 타겟 변수(합격여부)는 절대 채우면 안 됩니다!
before = df.isnull().sum().sum()

numeric_cols = df.select_dtypes(include='number').columns.tolist()
numeric_cols.remove('합격여부')   # 타겟 제외!

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

after = df.isnull().sum().sum()
print(f"✅ 결측값 처리 완료!")
print(f"   처리 전: {before}개  →  처리 후: {after}개")
print(f"\n📋 처리 후 결측값 현황:")
print(df.isnull().sum())


---
## 4️⃣ 탐색적 데이터 분석 (EDA)

> **EDA(Exploratory Data Analysis)** = 데이터를 다양한 각도로 뜯어보는 과정  
> "데이터와 친해지기" 단계라고 생각하면 됩니다!
>
> ✅ 분포는 어떤가? ✅ 이상치는 없나? ✅ 변수끼리 관계는?


In [ ]:
# ── 합격/불합격 그룹별 평균 비교 ───────────────────────────
# groupby() : 특정 기준으로 데이터를 묶어 집계합니다
group_mean = df.groupby('합격여부')[numeric_cols].mean().round(2)
group_mean.index = ['❌ 불합격(-1)', '✅ 합격(1)']

print("📊 합격 vs 불합격 그룹별 평균값 비교:")
group_mean.T.style.background_gradient(cmap='RdYlGn', axis=1)


In [ ]:
# ── 그룹별 평균 차이 시각화 ─────────────────────────────────
diff = (group_mean.loc['✅ 합격(1)'] - group_mean.loc['❌ 불합격(-1)']).sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in diff.values]
bars = ax.barh(diff.index, diff.values, color=colors,
               edgecolor='white', height=0.6)
ax.axvline(0, color='black', linewidth=1.5, linestyle='--')
ax.set_xlabel('합격 평균 - 불합격 평균')
ax.set_title('합격 vs 불합격 ─ 그룹 평균 차이\n(초록=합격이 더 높음 / 빨강=불합격이 더 높음)',
             fontsize=13, fontweight='bold')
for bar, val in zip(bars, diff.values):
    x_pos = val + 0.3 if val >= 0 else val - 0.3
    ha = 'left' if val >= 0 else 'right'
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f'{val:+.2f}', va='center', ha=ha, fontsize=9)
plt.tight_layout()
plt.show()
print("💡 두께_nm는 합격이 더 낮고, 전력_W도 합격이 더 낮습니다 → 주요 판별 요인!")


In [ ]:
# ── 각 수치형 컬럼의 분포 히스토그램 ───────────────────────
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    # 합격/불합격 각각 분포 그리기
    pass1  = df[df['합격여부'] ==  1][col]
    pass_1 = df[df['합격여부'] == -1][col]

    axes[i].hist(pass1,  bins=20, alpha=0.6, color='#2ecc71', label='합격(1)')
    axes[i].hist(pass_1, bins=20, alpha=0.6, color='#e74c3c', label='불합격(-1)')
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('값')
    axes[i].set_ylabel('빈도')
    axes[i].legend(fontsize=8)

# 남은 axes 숨기기
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('📊 각 센서 변수의 분포 (합격 vs 불합격)',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── 박스플롯(Box Plot)으로 이상치 탐지 ─────────────────────
# 박스플롯 읽는 법:
#   ├─ 박스 안 선: 중앙값(median)
#   ├─ 박스 위/아래: 75%, 25% 지점
#   └─ 점(•): 이상치(outlier)

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(x='합격여부', y=col, data=df,
                palette={1: '#2ecc71', -1: '#e74c3c'},
                ax=axes[i], width=0.5)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xticklabels(['불합격(-1)', '합격(1)'])
    axes[i].set_xlabel('')

for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('📦 박스플롯 - 합격/불합격별 센서값 분포 & 이상치',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── 바이올린 플롯 (분포 + 이상치 동시에) ───────────────────
# 바이올린 플롯 = 박스플롯 + 분포 모양을 함께 보여줌
key_cols = ['온도_섭씨', '전력_W', '진공도_mTorr', '두께_nm', '가스유량_slm', '처리시간_sec']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(key_cols):
    sns.violinplot(x='합격여부', y=col, data=df,
                   palette={1: '#2ecc71', -1: '#e74c3c'},
                   inner='box', ax=axes[i])
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].set_xticklabels(['불합격(-1)', '합격(1)'])
    axes[i].set_xlabel('')

plt.suptitle('🎻 바이올린 플롯 - 주요 변수 분포 비교',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── 상관관계(Correlation) 히트맵 ────────────────────────────
# 상관계수: -1 ~ +1 사이의 값
#   +1에 가까울수록 → 한쪽이 커지면 다른 쪽도 커짐
#   -1에 가까울수록 → 한쪽이 커지면 다른 쪽은 작아짐
#    0에 가까울수록 → 관계 없음

corr_matrix = df[numeric_cols + ['합격여부']].corr().round(2)

plt.figure(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # 위쪽 삼각형 숨기기
sns.heatmap(corr_matrix, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0,
            mask=mask,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 9})
plt.title('🔗 변수 간 상관관계 히트맵\n(숫자가 클수록 강한 양의 상관, 음수면 음의 상관)',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 합격여부와 가장 상관관계 높은 컬럼
target_corr = corr_matrix['합격여부'].drop('합격여부').sort_values(key=abs, ascending=False)
print("\n📊 합격여부와의 상관관계 순위:")
for col, val in target_corr.items():
    bar_str = '█' * int(abs(val) * 20)
    direction = '양(+)' if val > 0 else '음(-)'
    print(f"  {col:12s}: {val:+.3f}  {bar_str} {direction}")


In [ ]:
# ── 시간에 따른 센서값 변화 (시계열) ───────────────────────
df['측정시간'] = pd.to_datetime(df['측정시간'])
df_sorted = df.sort_values('측정시간').reset_index(drop=True)

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

key_time_cols = ['온도_섭씨', '전력_W', '두께_nm']
colors_line = ['#e74c3c', '#3498db', '#9b59b6']

for i, (col, color) in enumerate(zip(key_time_cols, colors_line)):
    axes[i].plot(df_sorted['측정시간'], df_sorted[col],
                 color=color, linewidth=1.2, alpha=0.8)

    # 불합격 지점 강조
    fail_mask = df_sorted['합격여부'] == -1
    axes[i].scatter(df_sorted.loc[fail_mask, '측정시간'],
                    df_sorted.loc[fail_mask, col],
                    color='#e74c3c', zorder=5, s=60,
                    edgecolors='black', linewidths=0.5,
                    label='불합격 지점')

    axes[i].set_ylabel(col, fontsize=11)
    axes[i].legend(loc='upper right', fontsize=9)
    axes[i].grid(True, alpha=0.3)

axes[-1].set_xlabel('측정 시간', fontsize=11)
axes[-1].tick_params(axis='x', rotation=30)
plt.suptitle('📈 시간에 따른 주요 센서값 변화\n(빨간 점 = 불합격 발생)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── 산점도 행렬 (Pair Plot) ─────────────────────────────────
# 변수끼리 2개씩 짝지어 산점도를 그려서 관계를 한눈에 파악!
key_pair_cols = ['온도_섭씨', '전력_W', '진공도_mTorr', '두께_nm', '합격여부']

g = sns.pairplot(df[key_pair_cols],
                 hue='합격여부',
                 palette={1: '#2ecc71', -1: '#e74c3c'},
                 plot_kws={'alpha': 0.6, 's': 30},
                 diag_kind='hist')
g.fig.suptitle('🔍 주요 변수 쌍별 산점도 (Pair Plot)',
               y=1.02, fontsize=14, fontweight='bold')

handles = [mpatches.Patch(color='#2ecc71', label='합격(1)'),
           mpatches.Patch(color='#e74c3c', label='불합격(-1)')]
g.fig.legend(handles=handles, loc='upper right',
             bbox_to_anchor=(1.0, 1.0), fontsize=11)
plt.show()


---
## 5️⃣ 데이터 전처리 & 특성 준비

> 머신러닝 모델에 데이터를 넣기 전에 **정리**가 필요합니다.  
> 1. 필요 없는 컬럼 제거  
> 2. X(입력 특성)와 y(타겟) 분리  
> 3. 학습 / 테스트 데이터 분리  
> 4. 스케일링(Scaling) – 단위가 다른 변수들을 같은 눈금으로 맞추기


In [ ]:
# ── X(특성)와 y(타겟) 분리 ──────────────────────────────────
# 타겟: 합격여부 (우리가 예측하고 싶은 값)
# 특성: 나머지 센서 값들

X = df[numeric_cols].copy()       # 입력 변수
y = (df['합격여부'] == 1).astype(int)  # 타겟: 1=합격, 0=불합격 (이진 변환)

print("📊 X (특성 행렬) 크기:", X.shape)
print("📊 y (타겟 벡터) 크기:", y.shape)
print(f"\n타겟 분포: 합격={y.sum()}개, 불합격={len(y)-y.sum()}개")
print("\n📋 특성 목록:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")


In [ ]:
# ── 학습/테스트 데이터 분리 ─────────────────────────────────
# 전체 데이터를 학습용 80% + 테스트용 20%로 나눕니다
# 💡 random_state : 실행할 때마다 같은 결과가 나오도록 고정

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20%를 테스트용으로
    random_state=42,      # 재현성을 위한 시드 고정
    stratify=y            # 합격/불합격 비율 유지
)

print("📦 데이터 분리 결과:")
print(f"   학습 데이터: {X_train.shape[0]}행  (전체의 80%)")
print(f"   테스트 데이터: {X_test.shape[0]}행  (전체의 20%)")

# 분리 비율 시각화
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

splits = {'학습 데이터': len(X_train), '테스트 데이터': len(X_test)}
colors_split = ['#3498db', '#e67e22']
axes[0].bar(splits.keys(), splits.values(),
            color=colors_split, edgecolor='white', width=0.5)
axes[0].set_title('데이터 분리 현황', fontsize=12, fontweight='bold')
axes[0].set_ylabel('데이터 수')
for bar, val in zip(axes[0].patches, splits.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 1, f'{val}개',
                 ha='center', fontsize=11, fontweight='bold')

# 학습/테스트 내 합격 비율
train_ratio = [y_train.sum(), len(y_train)-y_train.sum()]
test_ratio  = [y_test.sum(),  len(y_test)-y_test.sum()]
x_pos = np.arange(2)
width = 0.35
axes[1].bar(x_pos - width/2, train_ratio, width,
            label='학습', color='#3498db', alpha=0.8)
axes[1].bar(x_pos + width/2, test_ratio,  width,
            label='테스트', color='#e67e22', alpha=0.8)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(['합격(1)', '불합격(0)'])
axes[1].set_title('학습/테스트 내 합격 비율', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].set_ylabel('데이터 수')

plt.tight_layout()
plt.show()


In [ ]:
# ── 스케일링(Scaling) ────────────────────────────────────────
# 💡 왜 필요한가?
#   온도(300°C) vs 진동(0.5mm/s) → 단위가 너무 다름
#   모델이 큰 수를 가진 변수에 편향될 수 있어요
#   → StandardScaler: 평균=0, 표준편차=1로 맞춰줌

scaler = StandardScaler()

# 학습 데이터로 스케일러를 '학습'시키고 변환
X_train_scaled = scaler.fit_transform(X_train)
# 테스트 데이터는 학습된 스케일러로만 변환 (fit은 하지 않음!)
X_test_scaled  = scaler.transform(X_test)

# 스케일링 전/후 비교
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 스케일링 전
X_train.boxplot(ax=axes[0], vert=False)
axes[0].set_title('스케일링 전\n(단위가 모두 다름)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('값')

# 스케일링 후
pd.DataFrame(X_train_scaled, columns=X_train.columns).boxplot(ax=axes[1], vert=False)
axes[1].set_title('스케일링 후\n(모두 같은 기준으로 정규화)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('표준화된 값')
axes[1].axvline(0, color='red', linestyle='--', alpha=0.5)

plt.suptitle('🔧 StandardScaler 적용 전/후 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ 스케일링 완료!")


---
## 6️⃣ 머신러닝 모델 학습

> 두 가지 모델을 비교해 봅니다!
>
> | 모델 | 특징 | 장점 |
> |------|------|------|
> | **로지스틱 회귀** | 통계 기반 선형 모델 | 해석이 쉬움, 빠름 |
> | **랜덤 포레스트** | 여러 결정 트리의 투표 | 성능 우수, 이상치에 강함 |


In [ ]:
# ── 두 모델 동시 학습 ───────────────────────────────────────
# 1) 로지스틱 회귀
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

# 2) 랜덤 포레스트
rf_model = RandomForestClassifier(
    n_estimators=100,   # 결정 트리 100개
    max_depth=5,        # 각 트리의 최대 깊이
    random_state=42
)
rf_model.fit(X_train_scaled, y_train)

print("✅ 모델 학습 완료!")
print(f"\n🌲 랜덤 포레스트: 결정 트리 {rf_model.n_estimators}개 학습")
print(f"📈 로지스틱 회귀: 학습 반복 {lr_model.n_iter_[0]}회")


In [ ]:
# ── 학습/테스트 정확도 비교 ─────────────────────────────────
results = {}
for name, model in [('로지스틱 회귀', lr_model), ('랜덤 포레스트', rf_model)]:
    train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
    test_acc  = accuracy_score(y_test,  model.predict(X_test_scaled))
    results[name] = {'학습 정확도': train_acc, '테스트 정확도': test_acc}
    print(f"\n{'='*35}")
    print(f"🤖 {name}")
    print(f"   학습 정확도  : {train_acc:.4f} ({train_acc*100:.1f}%)")
    print(f"   테스트 정확도: {test_acc:.4f} ({test_acc*100:.1f}%)")
    overfitting = train_acc - test_acc
    if overfitting > 0.05:
        print(f"   ⚠️  과적합 의심 (차이: {overfitting:.4f})")
    else:
        print(f"   ✅ 과적합 없음 (차이: {overfitting:.4f})")

# 정확도 비교 시각화
fig, ax = plt.subplots(figsize=(9, 5))
models = list(results.keys())
train_accs = [results[m]['학습 정확도'] for m in models]
test_accs  = [results[m]['테스트 정확도'] for m in models]

x = np.arange(len(models))
width = 0.35
bars1 = ax.bar(x - width/2, train_accs, width, label='학습 정확도',
               color='#3498db', alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + width/2, test_accs,  width, label='테스트 정확도',
               color='#e67e22', alpha=0.85, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=12)
ax.set_ylabel('정확도(Accuracy)')
ax.set_ylim(0.85, 1.02)
ax.set_title('모델별 학습/테스트 정확도 비교', fontsize=13, fontweight='bold')
ax.legend()
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.4)
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.002,
            f'{bar.get_height():.3f}',
            ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 7️⃣ 모델 평가 (상세 분석)

> 정확도(Accuracy) 하나만으론 부족합니다!  
> 특히 **불균형 데이터**(합격 90% vs 불합격 10%)에서는  
> 혼동행렬, Precision, Recall 등을 함께 봐야 합니다.
>
> | 지표 | 의미 | 반도체 공정 해석 |
> |------|------|-----------------|
> | **Precision** | 불합격 예측 중 실제 불합격 비율 | 틀린 경보 얼마나 적은가 |
> | **Recall** | 실제 불합격 중 예측 성공 비율 | 불량품을 얼마나 잡아냈나 |
> | **F1-Score** | Precision과 Recall의 조화 평균 | 종합 성능 |


In [ ]:
# ── 혼동행렬 (Confusion Matrix) ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (name, model) in zip(axes, [('로지스틱 회귀', lr_model),
                                      ('랜덤 포레스트', rf_model)]):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                ax=ax, linewidths=2, linecolor='white',
                xticklabels=['불합격(0)', '합격(1)'],
                yticklabels=['불합격(0)', '합격(1)'],
                annot_kws={'size': 16, 'weight': 'bold'})

    acc = accuracy_score(y_test, y_pred)
    ax.set_title(f'{name}\n정확도: {acc:.1%}', fontsize=13, fontweight='bold')
    ax.set_xlabel('예측값', fontsize=11)
    ax.set_ylabel('실제값', fontsize=11)

    # 설명 추가
    ax.text(0, 2.35, '⬆ 행: 실제값  |  열: 예측값 →',
            fontsize=8, color='gray')

plt.suptitle('🎯 혼동행렬 (Confusion Matrix)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📖 혼동행렬 읽는 법:")
print("  [0,0]: 불합격을 불합격으로 맞춤 (True Negative)")
print("  [0,1]: 불합격인데 합격으로 예측 (False Positive) ← 위험!")
print("  [1,0]: 합격인데 불합격으로 예측 (False Negative)")
print("  [1,1]: 합격을 합격으로 맞춤 (True Positive)")


In [ ]:
# ── 분류 성능 리포트 ────────────────────────────────────────
print("=" * 55)
print("📋 로지스틱 회귀 분류 성능 리포트")
print("=" * 55)
print(classification_report(y_test, lr_model.predict(X_test_scaled),
                             target_names=['불합격(0)', '합격(1)']))

print("=" * 55)
print("📋 랜덤 포레스트 분류 성능 리포트")
print("=" * 55)
print(classification_report(y_test, rf_model.predict(X_test_scaled),
                             target_names=['불합격(0)', '합격(1)']))


In [ ]:
# ── ROC 커브 비교 ───────────────────────────────────────────
# ROC 커브: 임계값을 바꿔가며 True Positive Rate vs False Positive Rate
# AUC(Area Under Curve): 1에 가까울수록 좋은 모델!

fig, ax = plt.subplots(figsize=(8, 7))

for name, model, color in [('로지스틱 회귀', lr_model, '#3498db'),
                             ('랜덤 포레스트', rf_model, '#e74c3c')]:
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5,
        alpha=0.5, label='랜덤 모델 (AUC = 0.500)')
ax.fill_between(fpr, tpr, alpha=0.05, color='#e74c3c')

ax.set_xlabel('False Positive Rate (1 - 특이도)', fontsize=12)
ax.set_ylabel('True Positive Rate (민감도)', fontsize=12)
ax.set_title('📈 ROC 커브 비교\n(AUC가 1에 가까울수록 좋은 모델)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.01, 1.0)
ax.set_ylim(0.0, 1.01)

# 완벽한 분류기 표시
ax.annotate('완벽한 분류기', xy=(0, 1), xytext=(0.15, 0.9),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=10, color='gray')

plt.tight_layout()
plt.show()


In [ ]:
# ── 랜덤 포레스트: 특성 중요도 시각화 ─────────────────────
# 특성 중요도 = 각 변수가 예측에 얼마나 기여하는가
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=numeric_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors_imp = ['#e74c3c' if v >= feat_imp.quantile(0.7)
              else '#f39c12' if v >= feat_imp.quantile(0.4)
              else '#95a5a6' for v in feat_imp.values]

bars = ax.barh(feat_imp.index, feat_imp.values,
               color=colors_imp, edgecolor='white', height=0.6)
ax.set_xlabel('특성 중요도 (Feature Importance)', fontsize=12)
ax.set_title('🌲 랜덤 포레스트 - 변수 중요도\n(수치가 클수록 예측에 중요한 변수)',
             fontsize=13, fontweight='bold')

for bar, val in zip(bars, feat_imp.values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

from matplotlib.patches import Patch
legend_el = [Patch(facecolor='#e74c3c', label='상위 30% 중요 변수'),
             Patch(facecolor='#f39c12', label='중간 중요도'),
             Patch(facecolor='#95a5a6', label='낮은 중요도')]
ax.legend(handles=legend_el, loc='lower right')
ax.axvline(feat_imp.mean(), color='navy', linestyle='--', alpha=0.5)
ax.text(feat_imp.mean() + 0.001, 0, '평균', color='navy', fontsize=9)

plt.tight_layout()
plt.show()

print("\n📊 특성 중요도 상위 3개:")
top3 = feat_imp.sort_values(ascending=False).head(3)
for col, val in top3.items():
    print(f"  {col}: {val:.4f}")


In [ ]:
# ── 새로운 데이터로 예측해보기 ─────────────────────────────
# 실제 현장에서처럼 센서값 하나를 넣고 합격 여부를 예측!

new_data = pd.DataFrame({
    '온도_섭씨'    : [300.0],
    '압력_Pa'     : [1013.0],
    '가스유량_slm' : [50.0],
    '전력_W'      : [2000.0],
    '진공도_mTorr' : [5.0],
    '두께_nm'     : [100.0],
    '습도_pct'    : [45.0],
    '진동_mm_s'   : [0.50],
    '처리시간_sec' : [120.0],
    '냉각수온도_섭씨': [20.0]
})

# 스케일링 적용
new_scaled = scaler.transform(new_data)

# 예측
rf_pred  = rf_model.predict(new_scaled)[0]
rf_prob  = rf_model.predict_proba(new_scaled)[0]

print("🔮 새로운 공정 데이터 예측 결과")
print("=" * 40)
print("입력 센서값:")
for col, val in new_data.iloc[0].items():
    print(f"  {col:12s}: {val}")
print("=" * 40)
result = '✅ 합격' if rf_pred == 1 else '❌ 불합격'
print(f"\n예측 결과: {result}")
print(f"합격 확률 : {rf_prob[1]*100:.1f}%")
print(f"불합격 확률: {rf_prob[0]*100:.1f}%")

# 확률 게이지
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.barh(['불합격 확률', '합격 확률'],
        [rf_prob[0]*100, rf_prob[1]*100],
        color=['#e74c3c', '#2ecc71'], height=0.4, edgecolor='white')
ax.set_xlim(0, 100)
ax.set_xlabel('%')
ax.set_title('예측 확률 게이지', fontsize=12, fontweight='bold')
for i, val in enumerate([rf_prob[0]*100, rf_prob[1]*100]):
    ax.text(val + 1, i, f'{val:.1f}%', va='center', fontweight='bold')
plt.tight_layout()
plt.show()


---
## 🎓 수업 정리

### ✅ 오늘 배운 내용

| 단계 | 핵심 내용 |
|------|-----------|
| **데이터 탐색** | `head()`, `info()`, `describe()`, `value_counts()` |
| **결측값 처리** | `isnull()`, `fillna(mean())` |
| **EDA** | `groupby()`, `corr()`, 히스토그램, 박스플롯 |
| **전처리** | `train_test_split()`, `StandardScaler` |
| **모델링** | `LogisticRegression`, `RandomForestClassifier` |
| **평가** | 혼동행렬, ROC-AUC, 특성 중요도 |

### 💡 핵심 인사이트

> 1. **온도, 전력, 진공도, 두께** 가 불합격을 결정하는 주요 변수
> 2. 랜덤 포레스트가 로지스틱 회귀보다 불균형 데이터에 강인함
> 3. 불합격이 10%에 불과한 **불균형 데이터**에서는 Recall이 중요한 지표

### 🚀 더 나아가기
- `XGBoost`, `LightGBM` 등 앙상블 모델 시도
- `SMOTE`로 불균형 데이터 오버샘플링
- `GridSearchCV`로 하이퍼파라미터 최적화
